# Fase 2: Modelado Analítico y Chat Interactivo de Negocio
Este notebook muestra cómo consumimos el resultado estructural del modelo de LightGBM, finalizando con un panel interactivo que conecta al pipeline expuesto en FastAPI.

In [ ]:
import sys
import os

# Hot-Reloading de la arquitectura base
root_path = os.path.abspath(os.path.join(os.getcwd(), '..'))
if root_path not in sys.path:
    sys.path.append(root_path)

%load_ext autoreload
%autoreload 2

from src.train_mora import MoraModelTrainer
from src.config import settings
import pandas as pd

## 2.1 Simulando el Pipeline MLOps
En lugar de entrenar línea por línea ensuciando el cuaderno, llamamos a la clase compilada.

In [ ]:
# Notebook auxiliar: el entrenamiento reproducible vive en 0_Master_Pipeline.ipynb.
# Artefacto esperado: settings.DATA_PROCESSED_DIR / "abt.parquet"
# Modelo esperado: settings.MODELS_DIR / "modelo_mora.pkl"
print("Pipeline LightGBM disponible vía 0_Master_Pipeline.ipynb y src.train_mora")

## 2.2 Chat interactivo de negocio vía FastAPI

Este cuaderno es auxiliar. Para la entrega principal, usa la celda de preguntas del `0_Master_Pipeline.ipynb`. Si deseas probar la API, levanta `uvicorn src.main:app --reload` desde la raíz y ejecuta la celda siguiente.

In [ ]:
import requests
from IPython.display import display, Markdown

def consultar_experto_llm(pregunta: str) -> None:
    """Consume el endpoint POST /ask-analyst de FastAPI y renderiza la respuesta."""
    url = "http://localhost:8000/ask-analyst"
    payload = {"question": pregunta}
    
    try:
        response = requests.post(url, json=payload, timeout=60.0)
        response.raise_for_status()
        data_json = response.json()
        display(Markdown(f"**Usuario:** {data_json['question']}"))
        display(Markdown(f"**TUMIPAY RAG Advisor:**\n{data_json['respuesta']}"))
    except requests.exceptions.ConnectionError:
        display(Markdown("**Error:** No se pudo conectar a la API. Levanta `uvicorn src.main:app --reload` en el puerto 8000."))
    except Exception as e:
        display(Markdown(f"**Error inesperado:** {e}"))

mi_pregunta = "¿Cuáles son los segmentos de mayor riesgo y qué acciones recomienda la política de cobranza?"
consultar_experto_llm(mi_pregunta)